# 03 — Two-Tower Model (Colab GPU Run)

**Run on Colab with a GPU.**

This notebook trains a two-tower BPR retrieval model on the **full** Online
Retail II dataset and exports the resulting item and user embeddings so the
pipeline can load them at serving time.

Runtime: approximately 5–10 minutes on a Colab T4 GPU with 30 epochs.

## Setup

```python
# Mount Drive and set a project path, or clone the repo:
# !git clone https://github.com/<you>/uk-retail-recommender.git
# %cd uk-retail-recommender
# !pip install -e ".[dev]"
```


In [ ]:
"""Copy-paste this as the first cell of every notebook in a portfolio project.

It walks up from the notebook's current working directory to find the project
root (the folder containing requirements.txt + src/), chdirs there, and adds
it to sys.path. After this cell runs, the notebook can:

  - import from src.x.y      (would otherwise fail with ModuleNotFoundError)
  - read relative paths like 'data/processed/...' or 'docs/...' regardless of
    where the notebook was launched from (notebooks/, project root, anywhere)
  - work the same whether launched from Jupyter, VS Code, jupyter nbconvert,
    or Colab (assuming the project was extracted to a known location there)
"""

# fmt: off
import os, sys
from pathlib import Path

_cwd = Path.cwd()
_root = next(
    (p for p in [_cwd] + list(_cwd.parents)
     if (p / 'requirements.txt').exists() and (p / 'src').is_dir()),
    None,
)
assert _root, f'Could not find project root from {_cwd}. Open the notebook from inside the project tree.'
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
print(f'Project root: {_root}')
# fmt: on


In [ ]:
# Download full data if not already present
import subprocess, pathlib

if not pathlib.Path("data/raw/online_retail_II.xlsx").exists():
    subprocess.run(["python", "scripts/download_data.py"], check=True)


In [ ]:
import pandas as pd
from src.data.clean import clean_transactions
from src.data.interactions import build_interactions

# Load and clean the full dataset
raw = pd.read_excel("data/raw/online_retail_II.xlsx")
df = clean_transactions(raw)
inter = build_interactions(df)
print(f"Users: {len(inter.user_ids):,}   Items: {len(inter.item_ids):,}")


## Train the two-tower model

`TwoTowerModel(dim=64, epochs=30)` uses BPR with in-batch negatives.
GPU training is automatic when `torch.cuda.is_available()` is True on Colab.


In [ ]:
from src.retrieval.two_tower import TwoTowerModel
import torch

device_str = "CUDA" if torch.cuda.is_available() else "CPU"
print(f"Training on: {device_str}")

tt = TwoTowerModel(dim=64, epochs=30, lr=0.05, batch_size=512, seed=42)
tt.fit(inter)
print("Training complete.")
print(f"Final epoch loss: {tt.loss_history_[-1]:.4f}")


## Training loss curve

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, len(tt.loss_history_) + 1), tt.loss_history_, marker="o", ms=4)
ax.set_title("Two-tower BPR training loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("Mean batch loss")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Export embeddings

Export item and user embeddings aligned row-for-row to `inter.item_ids` /
`inter.user_ids`. The pipeline checks for `models/two_tower_item_emb.npy` at
startup and automatically loads it as an additional retrieval source when present.


In [ ]:
import numpy as np
from pathlib import Path

models_dir = Path("models")
models_dir.mkdir(exist_ok=True)

item_emb = tt.item_embeddings()   # shape (n_items, dim)
user_emb = tt.user_embeddings()   # shape (n_users, dim)

np.save(models_dir / "two_tower_item_emb.npy", item_emb)
np.save(models_dir / "two_tower_user_emb.npy", user_emb)

print(f"Saved item embeddings: {item_emb.shape}")
print(f"Saved user embeddings: {user_emb.shape}")
print("The pipeline will auto-load two_tower_item_emb.npy on next run.")


## Next steps

Download the exported `.npy` files from Colab and place them in the `models/`
directory of the local repo. Re-run `04_ranking_eval.ipynb` to see the two-tower
rung added to the evaluation ladder.
